# Does the CUDA graph pay off? (issue #122)

The exact-Hessian collocation callback is **CPU-dispatch-bound, not GPU-bound**. Measured on an
A100 at `maxiter=100`, per Hessian call:

| | |
|---|---|
| CPU dispatch | **1259 ms** |
| GPU compute | **156 ms** |
| GPU busy | **12.4%** of the call |

and the exact Hessian achieves **12.6%** of its Amdahl ceiling. Those being the same number is
the point: the shortfall *is* the idle fraction. The GPU work itself is healthy -- 58.5% of
device time is fp64 GEMM on tensor cores. There is simply too little of it against ~18,000 ATen
dispatches per call.

`torch.compile` cannot help: Dynamo fails to trace `vmap(jacfwd(jacrev(.)))` on doubly-wrapped
`GradTrackingTensor(BatchedTensor(...))` even with `backend="eager"` -- a tracing limitation, not
a missing compiler. A **CUDA graph** sidesteps tracing entirely: record the eager kernel stream
once, replay it as a single launch.

### What is already verified (on CPU, no GPU needed)

* The refactor is **value-preserving**: the first three Hessian value vectors from a real solve
  are **bit-identical** to `dev` across all 35,658 entries.
* Replay is checked `torch.equal` against eager on first capture, and any failure falls back to
  eager permanently rather than raising.

### What this notebook measures -- the part nobody has verified

**Whether replay actually removes the dispatch cost.** The estimate (~234 s -> ~80 s at
`maxiter=100`) assumes it drives dispatch to near zero. That is the normal outcome for this
signature, but it is an assumption until measured on a GPU.

Two independent A/Bs, each toggling one thing:

1. `TWIN4BUILD_CUDA_GRAPH` 0 vs 1 -- the graph itself.
2. `TWIN4BUILD_HESS_CHUNK_DIV` 1 vs 2 -- the chunking, whose default this branch also changes
   (the Hessian chunk is now sized from its own memory budget instead of `_deriv_chunk // 2`).

Both report **pooled alongside seconds**: a faster run that lands elsewhere on a flat ridge is
not a better one.

**Setup**: Runtime > Change runtime type > **GPU** (A100 preferred), then Run all. ~20-30 min.

In [ ]:
# Quieten the translator's per-type namespace warnings.  They are a known
# issue (#114), they number in the hundreds per model build, and with several
# model builds per A/B they bury the results completely.
import warnings

warnings.filterwarnings("ignore", message="Failed to parse namespace")
warnings.filterwarnings("ignore", message="Failed to parse ontology namespace")
warnings.filterwarnings("ignore", message='Neither "df", "filename", nor "uuid"')
# --- Setup (Colab-aware) ---------------------------------------------------
# Needs TWIN4BUILD_HESS_CHUNK_DIV, added alongside this notebook.
TWIN4BUILD_REF = "fix/issue-damper-ventilation-identifiability"

try:
    import twin4build as tb
except ImportError:
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         f"git+https://github.com/JBjoernskov/Twin4Build.git@{TWIN4BUILD_REF}"],
        check=True,
    )
    import twin4build as tb

# Fail loudly now rather than 30 minutes in.
import inspect

import twin4build.estimator._transcription as _tr

_src = inspect.getsource(_tr)
_missing = [n for n in ("TWIN4BUILD_HESS_CHUNK_DIV", "exact_hessian",
                        "boundary_state_init") if n not in _src]
if _missing:
    raise RuntimeError(
        "The installed twin4build is missing: " + ", ".join(_missing)
        + f"""
Installed at: {tb.__file__}
Install a ref that has these, then restart the runtime:
    pip install -q --force-reinstall --no-deps git+https://github.com/JBjoernskov/Twin4Build.git@{TWIN4BUILD_REF}
    (Colab: Runtime > Restart session, then re-run this cell)"""
    )

import functools
import os
import time

import numpy as np
import pandas as pd
import torch

DEVICES = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])


def hardware():
    import platform
    import re

    cpu = platform.processor() or platform.machine()
    try:
        with open("/proc/cpuinfo") as fh:
            for line in fh:
                if line.lower().startswith("model name"):
                    cpu = line.split(":", 1)[1].strip()
                    break
    except OSError:
        pass
    ram = None
    try:
        with open("/proc/meminfo") as fh:
            ram = int(re.search(r"[0-9]+", fh.readline()).group()) / 1e6
    except OSError:
        pass
    gpu = fp64_ratio = None
    if torch.cuda.is_available():
        pr = torch.cuda.get_device_properties(0)
        gpu = f"{pr.name} ({pr.total_memory / 1e9:.0f} GB, sm_{pr.major}{pr.minor})"
        # Datacenter parts (A100/V100, sm_70/80/90) run fp64 at 1/2 of fp32;
        # consumer parts at 1/32-1/64.  Recorded because it was the first
        # hypothesis for the exact-Hessian result and it needs to stay visible.
        fp64_ratio = "1/2 (datacenter)" if pr.major in (7, 8, 9) and pr.minor == 0 \
            else "1/32-1/64 (consumer) -- suspect"
    return {"cpu": cpu, "cores": os.cpu_count(),
            "torch_threads": torch.get_num_threads(),
            "ram_gb": round(ram, 1) if ram else None,
            "gpu": gpu, "fp64": fp64_ratio, "torch": torch.__version__}


HW = hardware()
for k, v in HW.items():
    print(f"{k:14s} {v}")
if not torch.cuda.is_available():
    print("\nNO GPU -- Part A still works; the GPU comparisons will be skipped.")

In [ ]:
import datetime
import importlib.util as _ilu
import pathlib

from dateutil import tz

import twin4build.examples.utils as utils
from twin4build.utils.rgetattr import rgetattr

# `twin4build/examples/full_workflow_example/` (packaged CSVs) is a PACKAGE that
# shadows the module `full_workflow_example.py`, so load the module by path.
import twin4build.examples as _ex_pkg

_p = pathlib.Path(_ex_pkg.__file__).parent / "full_workflow_example.py"
_spec = _ilu.spec_from_file_location("_fwe_runtime", _p)
_mod = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
fcn = _mod.fcn

STEP = 1200
START = [datetime.datetime(2023, 12, 2, tzinfo=tz.gettz("Europe/Copenhagen"))]
END = [datetime.datetime(2023, 12, 7, tzinfo=tz.gettz("Europe/Copenhagen"))]
N_WARMUP = 20


def build_model(device="cpu", dtype=torch.float64, tag="prof"):
    m = tb.Model(id=f"{tag}_{device}")
    m.load(semantic_model_filename=utils.get_path(
        ["estimator_example", "one_room_example_model.xlsm"]), fcn=fcn)
    m.to(device, dtype)
    return m


def build_parameters(model):
    c = model.components
    space, heater = c["office"], c["office_space_heater"]
    hc, cc = c["office_temperature_heating_controller"], c["office_co2_controller"]
    valve = c["office_space_heater_valve"]
    sup, exh = c["office_supply_damper"], c["office_exhaust_damper"]
    occ, wall = c["office_occupancy"], c["office_boundary_wall"]
    det = c["office_occupancy_detector"]
    return [
        (space, "thermal.C_air", 5e5, 1e4, 5e5),
        (space, "thermal.C_wall", 1e6, 1e5, 3e6),
        (wall, "C", 1e6, 1e4, 1e7),
        (space, "thermal.R_out", 0.5, 0.01, 1),
        (space, "thermal.R_in", 0.1, 0.01, 1),
        (wall, "R_a", 0.04, 1e-4, 1),
        (wall, "R_b", 0.04, 1e-4, 1),
        (space, "thermal.f_wall", 0.1, 0, 10),
        (space, "thermal.f_air", 0.1, 0, 10),
        (space, "thermal.Q_occ_gain", 100.0, 10, 200),
        (heater, "thermalMassHeatCapacity", 1e4, 1e3, 2e5),
        (heater, "UA", None, 1, 100),
        (hc, "kp", 0.005, 1e-5, 1, "private"),
        (cc, "kp", 0.0001, 1e-5, 1, "private"),
        ([hc, cc], "Ti", 30, 1, 300, "private"),
        ([hc, cc], "Td", 0, 0, 1, "private"),
        (valve, "waterFlowRateMax", 0.001, 1e-6, 0.1),
        (valve, "valveAuthority", 1, 0.4, 1),
        ([sup, occ.supply_damper], "a", 1, 1, 10, "shared"),
        ([sup, occ.supply_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([exh, occ.exhaust_damper], "a", 1, 1, 10, "shared"),
        ([exh, occ.exhaust_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([space, occ], "mass.V", 65, 50, 80, "shared"),
        ([space, occ], "mass.G_occ", 1e-6, 1e-6, 1e-5, "shared"),
        ([space, occ], "mass.m_inf", 0.001, 1e-4, 0.01, "shared"),
        (det, "threshold", 1.0, 0.02, 5.0),
    ]


def build_measurements(model):
    return [(model.components["office_valve_position_sensor"], 0.05 / 2),
            (model.components["office_temperature_sensor"], 0.1 / 2),
            (model.components["office_damper_position_sensor"], 0.05 / 2),
            (model.components["office_co2_sensor"], 30 / 2)]


COLLOC_OPTS = {"boundary_state_init": "rollout", "early_stopping": False}
print("model builders ready")

In [ ]:
import os
import time

import numpy as np
import pandas as pd

from twin4build.utils.rgetattr import rgetattr

SD = {"office_temperature_sensor": 0.05, "office_valve_position_sensor": 0.025,
      "office_damper_position_sensor": 0.025, "office_co2_sensor": 15.0}
WARM_ITERS = 5
COLLOC_ITERS = 300


def score(model):
    out, pooled = {}, 0.0
    for cid, sd in SD.items():
        c = model.components[cid]
        sim = c.output["measuredValue"].history()[:, 0, 0].detach().cpu().numpy()[N_WARMUP:]
        act = c.time_series_input.values[:, 0, 0].detach().cpu().numpy()[N_WARMUP:]
        r = float(np.sqrt(np.mean((sim - act) ** 2)))
        out[cid.replace("office_", "").replace("_sensor", "")] = r
        pooled += (r / sd) ** 2
    out["pooled"] = pooled
    return out


def _carry(entry, model):
    comps, attr, _x0, lo, hi = entry[:5]
    comp = comps[0] if isinstance(comps, list) else comps
    v = float(rgetattr(comp, attr).get().reshape(-1)[0])
    eps = 1e-9 * (hi - lo)
    return (comps, attr, min(max(v, lo + eps), hi - eps), lo, hi, *entry[5:])


def run_arm(device, env, maxiter=COLLOC_ITERS, tag=""):
    """Full exact-Hessian arm under a given environment override."""
    prev = {k: os.environ.get(k) for k in env}
    os.environ.update({k: str(v) for k, v in env.items()})
    try:
        model = build_model(device, tag=f"ab{tag}")
        sim = tb.Simulator(model)
        est = tb.Estimator(sim)
        params = build_parameters(model)
        meas = build_measurements(model)

        t0 = time.perf_counter()
        est.estimate(START, END, STEP, params, meas, n_warmup=N_WARMUP,
                     method=("scipy", "SLSQP", "ad"),
                     options={"maxiter": WARM_ITERS, "fast": True})
        p2 = [_carry(e, model) for e in params]
        opts = dict(COLLOC_OPTS)
        opts.update({"maxiter": maxiter, "exact_hessian": True})
        r = est.estimate(START, END, STEP, p2, meas, n_warmup=N_WARMUP,
                         method=("casadi", "ipopt", "ad", "collocation"), options=opts)
        wall = time.perf_counter() - t0

        seed = r.get("estimated_initial_state", {}) or {}
        model.set_save_simulation_result(flag=True)
        sim.simulate(step_size=STEP, start_time=START, end_time=END,
                     after_initialize=lambda: [model.get_component(c).set_state(x)
                                               for c, x in seed.items()])
        row = {"device": device, **env, "seconds": wall,
               "iterations": r["iterations"]}
        row.update(score(model))
        return row
    finally:
        for k, v in prev.items():
            if v is None:
                os.environ.pop(k, None)
            else:
                os.environ[k] = v


# Discard a throwaway arm first.  The FIRST arm in a session pays model-build,
# import and allocator warm-up that later arms do not: in the previous run the
# first CPU arm took 2152 s where an identical configuration later took 1081 s,
# which read as a spurious 2.00x "speedup" from a toggle that does nothing on
# CPU.  Timings are only comparable once that cost is already paid.
print("warm-up arm (discarded)...", flush=True)
try:
    _ = run_arm(DEVICES[0], {"TWIN4BUILD_CUDA_GRAPH": 0,
                             "TWIN4BUILD_HESS_CHUNK_DIV": 1},
                maxiter=20, tag="warm")
    print("  warm-up done", flush=True)
except Exception as exc:
    print(f"  warm-up failed (continuing): {type(exc).__name__}: {exc}", flush=True)

rows = []
for dev in DEVICES:
    for graph in (0, 1):
        env = {"TWIN4BUILD_CUDA_GRAPH": graph, "TWIN4BUILD_HESS_CHUNK_DIV": 1}
        print(f"  {dev} | exact | CUDA_GRAPH={graph}", flush=True)
        try:
            rows.append(run_arm(dev, env, tag=f"g{graph}{dev}"))
            print(f"    {rows[-1]['seconds']:.0f}s pooled={rows[-1]['pooled']:.2f}", flush=True)
        except Exception as exc:
            print(f"    FAILED: {type(exc).__name__}: {exc}", flush=True)
            rows.append({"device": dev, **env, "seconds": float("nan"),
                         "error": f"{type(exc).__name__}: {exc}"})

ab = pd.DataFrame(rows)
if "error" in ab.columns and ab["error"].notna().any():
    print()
    print("!!! SOME ARMS FAILED -- the table below is incomplete !!!")
    for _, r in ab[ab["error"].notna()].iterrows():
        print(f"    {r['device']} graph={r.get('TWIN4BUILD_CUDA_GRAPH')} {r['error']}")
ab

from twin4build.estimator._cuda_graph import capture_status

print()
print("=" * 68)
print("CUDA GRAPH CAPTURE STATUS -- did it capture, or silently fall back?")
print("=" * 68)
_st = capture_status()
if not _st:
    print("  (nothing recorded -- no runner was constructed)")
for line in _st:
    print("  " + line)
if not any("captured" in s for s in _st) and "cuda" in DEVICES:
    print()
    print("  *** NO CAPTURE SUCCEEDED ON CUDA.  A 1.0x result therefore means")
    print("      the graph never ran, NOT that replay failed to help. ***")

In [ ]:
ok = ab[ab["seconds"].notna()] if "seconds" in ab.columns else ab
print("=" * 68)
print("A/B 1 -- does the CUDA graph pay off?")
print("=" * 68)
for dev in DEVICES:
    off = ok[(ok.device == dev) & (ok.TWIN4BUILD_CUDA_GRAPH == 0)]
    on = ok[(ok.device == dev) & (ok.TWIN4BUILD_CUDA_GRAPH == 1)]
    if not len(off) or not len(on):
        continue
    s0, s1 = float(off.seconds.iloc[0]), float(on.seconds.iloc[0])
    p0, p1 = float(off.pooled.iloc[0]), float(on.pooled.iloc[0])
    print(f"  {dev:5s} {s0:7.1f}s -> {s1:7.1f}s  ({s0 / s1:5.2f}x)"
          f"   pooled {p0:6.2f} -> {p1:6.2f}")
    if dev == "cpu":
        print("        (cpu has nothing to capture -- any difference here is noise,")
        print("         and a LARGE difference would mean something is wrong)")
    if abs(p1 - p0) > 1e-6:
        print("        *** POOLED MOVED -- replay must be bit-identical, so a")
        print("            change here means the graph is NOT running the same")
        print("            computation.  Investigate before trusting the timing. ***")

print()
print("Reference, same problem, from gpu_benchmark_collocation.ipynb (A100):")
print("  cpu collocation_exact 1339.1s pooled 30.78 | cuda collocation_exact 784.6s pooled 32.91")
print()
print(f"CPU: {HW['cpu']} ({HW['cores']} cores) | GPU: {HW['gpu']} | fp64 {HW['fp64']}")
print("Quote these numbers WITH the hardware line above.")

In [ ]:
# A/B 2 -- the chunking default this branch changes, measured independently.
rows2 = []
for dev in DEVICES:
    for div in (2, 1):
        env = {"TWIN4BUILD_CUDA_GRAPH": 0, "TWIN4BUILD_HESS_CHUNK_DIV": div}
        print(f"  {dev} | exact | HESS_CHUNK_DIV={div} (graph off)", flush=True)
        try:
            rows2.append(run_arm(dev, env, tag=f"c{div}{dev}"))
            print(f"    {rows2[-1]['seconds']:.0f}s pooled={rows2[-1]['pooled']:.2f}", flush=True)
        except Exception as exc:
            print(f"    FAILED: {type(exc).__name__}: {exc}", flush=True)

chunk = pd.DataFrame(rows2)
display(chunk)
for dev in DEVICES:
    a = chunk[(chunk.device == dev) & (chunk.TWIN4BUILD_HESS_CHUNK_DIV == 2)]
    b = chunk[(chunk.device == dev) & (chunk.TWIN4BUILD_HESS_CHUNK_DIV == 1)]
    if len(a) and len(b):
        s2, s1 = float(a.seconds.iloc[0]), float(b.seconds.iloc[0])
        print(f"  {dev:5s} div2 {s2:7.1f}s -> div1 {s1:7.1f}s ({s2 / s1:.2f}x)"
              f"   pooled {float(a.pooled.iloc[0]):.2f} -> {float(b.pooled.iloc[0]):.2f}")
print()
print("Chunking is EXACT -- it only splits the vmap -- so pooled should be")
print("unchanged except where a different iterate is reached by chance.")

## How to read it

**The graph is a pure speed change.** Replay executes the recorded kernels, so `pooled` must not
move. If it does, the capture is not running the computation we think it is, and the timing is
worthless until that is explained. The verdict cell checks this rather than leaving it to the
reader.

**CPU is the control.** There is nothing to capture on CPU, so `TWIN4BUILD_CUDA_GRAPH` should
make no difference there. A large CPU difference would indicate the toggle is doing something
unintended.

**Expect the payoff to be bounded by the Hessian's share.** The Hessian is 76% of wall-clock and
its dispatch is ~89% of its own cost, so perfect replay caps out near 3x on the arm -- not the
~20x the Amdahl ceiling suggests, because that ceiling measures how much work *could* move to the
GPU, not how well the GPU runs it.

**If the graph fails to capture**, the runner falls back to eager and logs one warning, so the
`CUDA_GRAPH=1` arm would simply match the `=0` arm. That is a legitimate outcome to report, not
a broken run -- check the output for the fallback warning before concluding the graph "did not
help".